# Entrenamiento del modelo final

Entrena el modelo final con **todo** `train_data.jsonlines` y lo guarda en `modelo_final.joblib`. Contiene **solo** el código necesario para replicarlo. La submission la genera `predict.ipynb`.

**Modelo final:** promedio 50/50 de las probabilidades de dos modelos que codifican las categóricas de forma distinta:
- **CatBoost** (parámetros por defecto) con las categóricas crudas, la probabilidad de un modelo de texto (título + garantía) y features numéricas de la publicación;
- **HistGradientBoosting** con las mismas features numéricas y de texto, y el vendedor, la categoría y sus combinaciones codificados con `TargetEncoder`.

Los dos modelos usan además la **duración de la publicación**, y como probabilidad de texto el **promedio de una regresión logística y de BETO** (BERT en español con *fine-tuning*). Accuracy en CV estratificada de 5 folds (semilla 42): CatBoost 0,9431, HistGB 0,9425, **promedio 0,9435** (sin BETO: 0,9420) (comparación de modelos en `trabajo_realizado/13_comparacion_modelos.ipynb`; desarrollo en `trabajo_realizado/laboratorio.ipynb` y `BITACORA.md`). El promedio se hace en `predict.ipynb`; acá se entrenan y guardan los dos modelos.

Todas las funciones y transformers vienen de `features.py`.

**Tiempo:** ~100 min, casi todo el *fine-tuning* de los 5 BETO en la GPU de la Mac (MPS). Correr con la compu enchufada y sin apps pesadas. Los pesos quedan en `modelos_beto/` y `predict.ipynb` los carga.

In [1]:
CON_BETO = True  # False reproduce la submission de respaldo: modelo sin BETO (con_beto=False), guarda modelo_final_p7c_duracion.joblib

import pandas as pd
import joblib
import warnings
import transformers

from texto_beto import oof_beto

from features import (
    aplanar_anidadas, agregar_features_contenido, agregar_categoricas_crudas, agregar_combos,
    armar_pipeline_final, armar_pipeline_histgb_combos, INDICES_CAT,
)

transformers.logging.set_verbosity_error()  # oculta los avisos de carga de BETO (LOAD REPORT)

/Users/nicolasscravaglieri/Downloads/kaggle-ap-2026/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Carga de datos

`y` vale 1 si la publicación es `new` y 0 si es `used`.

In [2]:
train = pd.read_json("data/train_data.jsonlines", lines=True)
y = (train["condition"] == "new").astype(int)

## 2. Features determinísticas

Estas cuatro funciones **no aprenden nada de los datos** (calculan columnas fila por fila). Por eso van afuera de los `Pipeline`, y `predict.ipynb` aplica exactamente las mismas al test:

- `aplanar_anidadas`: abre las columnas anidadas (envío, dirección del vendedor, fotos, atributos, métodos de pago, tags) en columnas planas, conteos y flags de presencia.
- `agregar_features_contenido`: fotos (resolución de la primera, % cuadradas, % con proporción de cámara 4:3, cantidad de tamaños distintos), fechas (hora de inicio, días hasta la última actualización), la garantía como texto (`warranty_texto`) y la **duración de la publicación** (`duracion_dias` = `stop_time` − `start_time`). Las publicaciones que duran 60 días + unos segundos son casi todas `used` (publicaciones creadas por primera vez; ver `trabajo_realizado/18_barrido_campos.ipynb`).
- `agregar_categoricas_crudas`: copia como texto las 6 categóricas que CatBoost usa crudas (`seller_id`, `category_id`, `listing_type_id`, `buying_mode`, `shipping_mode`, `seller_state`).
- `agregar_combos`: combinaciones de categóricas como texto (vendedor × categoría, vendedor × tipo de publicación, categoría × rango de precio, …) más algunas categóricas sueltas. Las usa el HistGB. Ejemplo: un vendedor que vende celulares nuevos y ropa usada tiene 50% `new` en total, pero la combinación vendedor × categoría separa los dos casos.

In [3]:
train = aplanar_anidadas(train)
train = agregar_features_contenido(train)
train = agregar_categoricas_crudas(train)
train = agregar_combos(train)

## 3. Probabilidad de texto de BETO

BETO es un BERT pre-entrenado en español. Se le hace *fine-tuning* para `new` vs `used` con la entrada `[CLS] título [SEP] garantía [SEP]` (2 épocas, AdamW, *learning rate* 2e-5 con *warmup*; `texto_beto.py`).

Igual que con la logística de texto, cada fila del train tiene que recibir la probabilidad de un modelo **que no la vio**: si no, BETO casi memoriza el train y los dos modelos aprenderían a confiar en una probabilidad demasiado buena, que en el test no se repite. Por eso `oof_beto` entrena **5 modelos**, cada uno sin un fold de `StratifiedKFold(5, shuffle=True, random_state=42)`, y a cada fila le asigna la probabilidad del modelo que no la tuvo en su entrenamiento. Guarda los pesos de los 5 en `modelos_beto/fold_k/`: `predict.ipynb` promedia los 5 sobre el test. Todas las semillas (Python, numpy, torch, transformers) están fijas en 42.

In [4]:
if CON_BETO:
    train["prob_beto_new"] = oof_beto(train["title"], train["warranty_texto"], y, "modelos_beto")

## 4. Los dos modelos

En los dos `Pipeline`, todo lo que aprende de los datos está adentro:
- `CapeoPorMoneda` recorta `price`/`base_price` al percentil 99,5 de su moneda, y `agregar_logs` calcula `log_price` y `log_available_quantity`.
- `prob_texto_new`: promedio de dos probabilidades de `new` del texto del título y de la garantía (`PromedioConBeto`): una regresión logística (`PrediccionOOF`; en el train se calcula con una CV interna, para que cada fila reciba la probabilidad de un modelo que no la vio) y BETO (`prob_beto_new`, sección 3). Las dos se equivocan en filas distintas, así que su promedio es mejor que cada una (`trabajo_realizado/29_texto_combinado.ipynb`).

**CatBoost** (`armar_pipeline_final`): 24 numéricas + `duracion_dias` + TF-IDF del título (500) + `prob_texto_new` + las 6 categóricas crudas. Codifica las categóricas con *ordered target statistics*: cada fila se codifica con el % de `new` de las filas **anteriores** de su categoría en un orden al azar, así que nunca usa su propia etiqueta. Las categóricas se le indican en el `fit` (`modelo__cat_features=INDICES_CAT`), por una incompatibilidad de CatBoost con `clone` de scikit-learn.

**HistGradientBoosting** (`armar_pipeline_histgb_combos`): one-hot de 4 categóricas + las mismas numéricas, TF-IDF y `prob_texto_new` + `TargetEncoder` sobre `seller_id`, `category_id` y las combinaciones (*cross-fitting*: cada fila se codifica con el % de `new` de su categoría en los otros 4/5 del train, nunca con su propia etiqueta) + cantidad de publicaciones y precio promedio del vendedor (sin target) + `duracion_dias`.

In [5]:
modelo_catboost = armar_pipeline_final(con_beto=CON_BETO)
modelo_histgb = armar_pipeline_histgb_combos(con_beto=CON_BETO)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    modelo_catboost.fit(train, y, modelo__cat_features=INDICES_CAT)
    modelo_histgb.fit(train, y)

## 5. Guardado

Los dos modelos van juntos en un diccionario. `predict.ipynb` los carga y promedia sus probabilidades. Los pesos de BETO ya quedaron guardados en `modelos_beto/` (sección 3).

In [6]:
archivo_modelo = "modelo_final.joblib" if CON_BETO else "modelo_final_p7c_duracion.joblib"
joblib.dump({"catboost": modelo_catboost, "histgb": modelo_histgb}, archivo_modelo)

['modelo_final.joblib']